# PARC2026 — 73 M3 guarded training-adapter preflight

72d PASS後に、実benchmarkを開始せず、D10 canonical sampling schedule・72d確定batch設定・正順/逆順run matrixを結合して検証します。ここではGPU学習を一切開始しません。


In [ ]:
import json, os, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_m3_runner'
PIN = 'af538477f4c73a08ca73757b656bb94123999b0e'
URL = 'https://github.com/yu37330/py_AI.git'
if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', '--force', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('73 preflight code:', got, flush=True)

DRIVE = Path('/content/drive/MyDrive/parc2026-cache')
DATASET = DRIVE / 'datasets/lerobot_libero_plus_v3_train'
BATCH_SUMMARY = DRIVE / 'model-benchmark-v1/m3_batch_probe_summary.json'
EXPECTED_HASH = '73ed0d3b0c5e73c745c0aa2e81517ce1fa40240c75f9eb040d65b6876ba08239'
manifest_candidates = [
    DRIVE / 'pi05-ablation-group-aware-v2/dataset_ablation_manifests_v2_group_aware/V2_SQRT_BALANCED_RAW.json',
    ROOT / 'outputs/dataset_ablation_manifests_v2_group_aware/V2_SQRT_BALANCED_RAW.json',
]
MANIFEST = None
for candidate in manifest_candidates:
    if not candidate.is_file():
        continue
    data = json.loads(candidate.read_text(encoding='utf-8'))
    if data.get('episode_ids_sha256') == EXPECTED_HASH:
        MANIFEST = candidate
        break
if MANIFEST is None:
    raise FileNotFoundError('Exact D10 group-aware manifest not found; do not regenerate/reselect it')
for required in (DATASET / 'meta/info.json', BATCH_SUMMARY):
    if not required.is_file():
        raise FileNotFoundError(required)
summary = json.loads(BATCH_SUMMARY.read_text(encoding='utf-8'))
if summary.get('status') != 'READY_FOR_M3_RUNNER_IMPLEMENTATION':
    raise RuntimeError(f'72d gate not ready: {summary.get("status")}')
if summary.get('benchmark_training_started') is not False:
    raise RuntimeError('72d summary unexpectedly claims benchmark training started')

SCHEDULE_DIR = DRIVE / 'model-benchmark-v1/schedules'
SCHEDULE_DIR.mkdir(parents=True, exist_ok=True)
SCHEDULE = SCHEDULE_DIR / 'm3_equal_data_seed_20260906.json'
subprocess.run([
    'python', '-u', str(REPO / 'tools/benchmark/m3_sampling_schedule.py'),
    '--manifest', str(MANIFEST), '--dataset-root', str(DATASET),
    '--seed', '20260906', '--out', str(SCHEDULE),
], cwd=str(REPO), check=True)

OUT = DRIVE / 'model-benchmark-v1/m3_training_adapter_preflight.json'
subprocess.run([
    'python', '-u', str(REPO / 'tools/benchmark/m3_training_adapters.py'),
    '--batch-summary', str(BATCH_SUMMARY), '--equal-data-schedule', str(SCHEDULE),
    '--results-root', str(DRIVE / 'model-benchmark-v1'),
    '--seed', '20260906', '--out', str(OUT),
], cwd=str(REPO), check=True)
plan = json.loads(OUT.read_text(encoding='utf-8'))
if plan.get('status') != 'READY_FOR_GUARDED_SMOKE_IMPLEMENTATION':
    raise RuntimeError(plan)
if plan.get('run_count') != 12 or plan.get('training_started') is not False:
    raise RuntimeError(plan)
print(json.dumps({
    'status': plan['status'],
    'run_count': plan['run_count'],
    'orders': plan['orders'],
    'equal_data_schedule_sha256': plan['equal_data_schedule_sha256'],
    'training_started': plan['training_started'],
    'out': str(OUT),
}, indent=2), flush=True)
print('=== 73 COMPLETE ===', flush=True)
print('No training started. Next: guarded per-model worker smoke implementation.', flush=True)
